# Test retrieving relevant content to user query

- Collect all vectors and metadata in one place

In [4]:
import pandas as pd
from sentence_transformers import SentenceTransformer
from nesta_ds_utils.loading_saving import S3
from time import time

from discovery_child_development.getters.openalex import get_sentence_embeddings

2024-07-05 09:54:22,511 - botocore.credentials - INFO - Found credentials in environment variables.
2024-07-05 09:54:23,003 - datasets - INFO - PyTorch version 2.1.2 available.


In [163]:
from discovery_child_development.utils.openai_utils import client

In [ ]:
model = SentenceTransformer("all-MiniLM-L6-v2")

from discovery_child_development import PROJECT_DIR, logging, S3_BUCKET
ENRICHED_DATA_DIR = PROJECT_DIR / 'outputs/enrichments'

VECTORS_PATH = "data/outputs/vectors/"

In [5]:
vector_files =[
    "sentence_vectors_384_labelled.parquet",
    "sentence_vectors_openalex_384_labelled.parquet",
    "sentence_vectors_patents_384_labelled.parquet",
    "sentence_vectors_gtr_384_labelled.parquet",
    "sentence_vectors_crunchbase_384_labelled.parquet",
]

In [44]:
vect_df = []
for vector_file in vector_files:
    vect_df.append(
        get_sentence_embeddings(
            s3_bucket=S3_BUCKET,
            filepath=VECTORS_PATH,
            filename=vector_file,
            id="id",
        )
       .reset_index()
       # Simplify the id by removing https
       .assign(id=lambda df: df["id"].apply(lambda x: x.split("/")[-1]))
       .set_index("id")        
    )

In [45]:
vect_df = pd.concat(vect_df)

In [46]:
len(vect_df)

125176

In [13]:
query = "Helping early childhood progressionals with saving admin time"
queries = [query]
query_vect = model.encode(queries, show_progress_bar=False)

In [18]:
# use cosine similarity to find the most similar sentence for query_vect in vect_df
t0 = time()
from sklearn.metrics.pairwise import cosine_similarity
cosine_similarities = cosine_similarity(query_vect, vect_df["miniLM_384_vector"].tolist())
# find the index of the n most similar
n = 5
most_similar = cosine_similarities[0].argsort()[-n:][::-1]

In [20]:
vect_df.iloc[most_similar]

,miniLM_384_vector
id,
8ab84b25-87e8-49b7-bd2d-abfcd7172b91,"[-0.045013513, 0.01989712, 0.009523964, 0.0618..."
b9f5878c-5128-44d9-9696-e0993beb4e20,"[-0.005000229, 0.071935825, 0.047169726, 0.061..."
161f30d0-0ba7-4051-844b-91bdbccb36ef,"[-0.024793144, 0.027843386, 0.027431106, 0.064..."
https://openalex.org/W2758233828,"[-0.06352416, 0.034508664, 0.0012036905, 0.075..."
https://openalex.org/W3109585718,"[-0.037461173, 0.031438757, -0.025601769, -0.0..."


## Load all the metadata

In [ ]:
from discovery_child_development.analysis.initial_results import utils

In [36]:
patents_df = pd.read_csv(utils.PROJECT_DIR / "outputs/data/tables/patents_final.csv")
openalex_df = pd.read_csv(utils.PROJECT_DIR / "outputs/data/tables/openalex_final.csv")
gtr_df = pd.read_csv(utils.PROJECT_DIR / "outputs/data/tables/gtr_final.csv")
crunchbase_df = pd.read_csv(utils.PROJECT_DIR / "outputs/data/tables/crunchbase_final.csv")

In [97]:
cols = ["id", "text", 'dataset', 'topic', 'major_category', 'minor_category', 'topic_code', 'url']
data_df = (
    pd.concat([
        patents_df[cols],
        openalex_df[cols],
        gtr_df.assign(dataset="gtr")[cols],
        crunchbase_df.assign(dataset="crunchbase")[cols],
    ], ignore_index=True)
    .drop_duplicates(subset=["id"])
    .drop_duplicates(subset=["text"])
)

In [101]:
data_df.sample()

,id,text,dataset,topic,major_category,minor_category,topic_code,url
5853,US-2015319478-A1,"Pattern code recognition multimedia playback apparatus, and method for driving same. The present disclosure describes a sensing technology. A pattern code recognition multimedia playback apparatus and driving method therefor according to some embodiments recognize pattern code by using pattern c...",patents,"Preschool, Expressive arts and design, Infancy, Games","General, Development & learning, Child care & preschool","Preschool, Expressive arts and design, Infancy, Games","infancy, games, arts, preschool",https://patents.google.com/patent/US2015319478A1


In [49]:
ids = set(data_df.id.to_list())

In [53]:
vect_ids = set(vect_df.reset_index().id.to_list())

In [60]:
# make all text into one big string
text = " ".join(data_df.text.fillna("").to_list())
# calculate number of words
num_words = len(text.split())

In [61]:
num_words

23556153

In [64]:
# check that all ids in data_df are in vect_df
check_ids = (ids - vect_ids)
extra_vectors_df = crunchbase_df.query("id in @check_ids")

sentence_vectors_384 = model.encode(extra_vectors_df.text.to_list(), show_progress_bar=True)
vectors_as_list = [list(vec) for vec in sentence_vectors_384]
extra_vector_df = pd.DataFrame({"id": extra_vectors_df.id.to_list(), "miniLM_384_vector": vectors_as_list})


Batches: 100%|██████████| 32/32 [00:03<00:00, 10.23it/s]


In [ ]:
vectors_df = pd.concat([
    vect_df.reset_index(),
    extra_vector_df
], ignore_index=True).drop_duplicates(subset=["id"])

In [74]:
missing_ids = (ids - set(vectors_df.id.to_list()))

In [75]:
# check that all ids in data_df are in vect_df
extra_vectors_df = openalex_df.query("id in @missing_ids")

sentence_vectors_384 = model.encode(extra_vectors_df.text.to_list(), show_progress_bar=True)
vectors_as_list = [list(vec) for vec in sentence_vectors_384]
extra_vector_df = pd.DataFrame({"id": extra_vectors_df.id.to_list(), "miniLM_384_vector": vectors_as_list})


Batches: 100%|██████████| 1/1 [00:00<00:00, 19.87it/s]


In [76]:
vectors_df = pd.concat([
    vectors_df,
    extra_vector_df
], ignore_index=True).drop_duplicates(subset=["id"])

In [77]:
(ids - set(vectors_df.id.to_list()))

set()

## Organise data

In [114]:
data_df[data_df.text.isnull()]

,id,text,dataset,topic,major_category,minor_category,topic_code,url
99321,6aed65ea-6208-9b03-95e9-b77ba6d5d5d1,NaN,crunchbase,Games,General,Games,games,NaN


In [116]:
len(data_df)

104637

In [121]:
_vectors_df = vectors_df.query("id in @data_df.id.to_list()").reset_index()
len(_vectors_df)

104637

In [140]:
amounts_df = pd.concat([
    gtr_df[['id', 'amount']],
    crunchbase_df[['id', 'total_funding_gbp']].rename(columns={"total_funding_gbp": "amount"}),
], ignore_index=True)


In [171]:
cols = ["id", "text", 'dataset', 'topic', 'major_category', 'minor_category', 'topic_code', 'country_code', 'url']
data_df = (
    pd.concat([
        patents_df[cols],
        openalex_df[cols],
        gtr_df.assign(dataset="gtr").assign(country_code='GB')[cols],
        crunchbase_df.assign(dataset="crunchbase")[cols],
    ], ignore_index=True)
    .drop_duplicates(subset=["id"])
    .drop_duplicates(subset=["text"])
    .merge(amounts_df, on="id", how="left")
    .merge(
        crunchbase_df[['id', 'name']].rename(columns={"name": "_id"}),
        on="id",
        how="left"
    )
    .assign(_id = lambda df: df["_id"].fillna(df["id"]))
)

In [173]:
data_df.to_csv(utils.PROJECT_DIR / "outputs/data/tables/full_data_final.csv", index=False)

In [162]:
_vectors_df.drop(columns=["index"]).to_parquet(utils.PROJECT_DIR / "outputs/data/tables/full_vectors_final.parquet", index=False)

In [190]:
# export as txt file
data_df.to_csv(utils.PROJECT_DIR / "outputs/data/tables/full_data_final.txt", sep="\t", index=False)

## Test retrieving

In [130]:
from sklearn.metrics.pairwise import cosine_similarity
pd.set_option('display.max_colwidth', 300)

_df = data_df.dropna(subset=["major_category"])
_df = _df[_df.major_category.str.contains("Technology")]

query = "Helping early childhood progressionals with saving admin time"
n = 10
def find_most_similar(query):
    queries = [query]
    query_vect = model.encode(queries, show_progress_bar=False)        
    cosine_similarities = cosine_similarity(query_vect, _vectors_df["miniLM_384_vector"].tolist())   

    return (
        _vectors_df
        .assign(similarity=cosine_similarities[0])
        # .query("id in @_df.id.to_list()")
        .sort_values("similarity", ascending=False)
        .iloc[0:10]
        .merge(
            data_df,
            on="id",
            how="left"
        )
    )    


In [180]:
query = "Company supporting early years professionals in communicating with parents"
docs = find_most_similar(query)[['_id', 'text', 'topic', 'similarity']]
docs

,_id,text,topic,similarity
0,Parent And Child Empowerment Organisation Community Interest Company,"This is one of the world's most advanced programs for supporting parents with infants and young children.. This is one of the world's most advanced programs for supporting parents with infants and young children. Instead of having to depend on experts to advise and guide them, parents are give...","Social services, Community, Infancy",0.673799
1,W2534993462,Engaging Parents in Early Childhood Education: Perspectives of Childcare Providers. Successful engagement of parents in early childhood education has significant implications for a growing child's well‐being and success. This qualitative study analyzes the perspectives of 14 North Carolina child...,Social services,0.669772
2,W2903425802,Perspectives on Parent-Educator Communication Practices in the Early Childhood Center. The purpose of this case study is to describe parent-early childhood educator (ECE) communication in a Quebec childcare center. Communication practices are first described and analyzed in terms of the percepti...,Preschool,0.658839
3,W2772763741,What's the Story?: Exploring Parent–Teacher Communication through ePortfolios. ELECTRONIC PORTFOLIOS (ePORTFOLIOS) ARE a relatively new phenomenon in early childhood education (ECE) with minimal existing research available on their use and effectiveness as a learning and communication tool in EC...,Internet,0.633108
4,Parent,"Revolutionizing and simplifies childcare management, enriches children’s early education, and streamlines communication with parents.. Parent™ is an all-in-one childcare management application for everything you need including attendance tracking, events with automatic reminders, daily reports, ...",Operations,0.627372
5,Parent,"Parent Co. is a thoughtful digital tool for parents and a place where they share stories, ideas, and inspirations about modern parenthood.","Mobile, Parenting, Community",0.627219
6,W2969380716,"Building Professional Capacity to Strengthen Parent/Professional Relationships in Early Intervention: The FAN Approach. A strong relationship between parents and professionals is essential to successful early intervention. Yet, programs struggle to engage families in services. This article descr...","Community, Social services, Personal social emotional, Special educational needs",0.619097
7,W4385307434,Establishing a Comprehensive &amp; Coherent Definition of Parent Responsiveness for Enhanced Multidisciplinary Team Communication in Early Intervention. Abstract Date Presented 04/21/2023 This systematic review of multidisciplinary literature yielded a comprehensive definition of parent responsi...,"Communication and language, Inclusion, Infancy",0.615631
8,W2982190716,"Talk Around Town: A Mobile Phone Application to Support Parent–Child Talk in the Community. Supports are largely absent and tools are scarce to assist parents receiving home-based early intervention services (e.g., Early Head Start) with enriching the language learning environments of young chil...","Community, Mobile, Social services",0.612351
9,iQuriousKids Inc.,Online platform to connect busy parents with kids activity providers to help parents unleash their children's unique potential,"Parenting, Operations",0.611109


In [186]:
texts = 'ID: ' + docs['_id'] + ' | TEXT: ' + docs['text']
abstract_texts = "\n\n".join(texts.to_list())

gpt_message = f"You are a designer working on a new product for early childhood professionals. You want to design a promising technological intervention based on the retrieved results. You have access to a dataset of {len(docs)} documents related to early childhood professionals and technology. Suggest an innovative technological intervention that could help early childhood professionals in this way: {query} \n\n The documents you can get inspiration from are the following (only use information from documents that are actually relevant): {abstract_texts}\n\n"
    

messages = [
    {
        "role": "user",
        "content": gpt_message,
    }
]
chatgpt_output = client.chat.completions.create(
    model = "gpt-4o-2024-05-13",
    messages=messages,
    temperature=0.6,
    max_tokens=None,
)

2024-07-05 15:35:54,721 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


In [187]:
output = chatgpt_output.choices[0].message.content

In [188]:
print(output)

Based on the insights derived from the relevant documents, an innovative technological intervention for early childhood professionals to support communication with parents could be a **"Comprehensive Parent-Professional Communication Hub"**. This platform would integrate multiple features tailored to enhance and streamline communication between early childhood professionals and parents.

### Key Features:

1. **ePortfolios**:
   - Inspired by the document ID W2772763741, the platform would include electronic portfolios (ePortfolios) for each child, where educators can upload daily updates, developmental milestones, photos, and videos. This feature facilitates ongoing communication and allows parents to stay informed about their child's progress and activities.

2. **Real-Time Communication and Notifications**:
   - Drawing from document ID Parent, the platform would incorporate real-time communication tools such as instant messaging and notifications. Parents can receive immediate upda